# 4. Análisis univariado

Se mira cada variable por separado: primero la variable objetivo y luego las 10 predictoras.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from utils import *

set_style()
np.random.seed(SEED)
df = load_data()

## 4.1 Variable objetivo

In [ ]:
pct = df[TARGET].value_counts(normalize=True)[CLASS_ORDER] * 100
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.barh(CLASS_ORDER, pct.values, height=0.6, color=[CLASS_COLORS[k] for k in CLASS_ORDER])
for y, v in enumerate(pct.values):
    ax.text(v + 1, y, f"{v:.1f} %", va="center")
ax.set_xlim(0, 85); ax.set_xlabel("% de plantas")
ax.set_title("Distribución de solar_aptittude_class")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

El desbalance es severo: **75.1 % Alta, 21.7 % Media y 3.2 % Baja** (1 892 plantas). Un modelo que siempre diga «Alta» acertaría 3 de cada 4 veces, así que la exactitud engaña: hay que evaluar con **F1 macro y recall por clase**.

## 4.2 Predictoras

### Histogramas

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(12, 12))
for ax, col in zip(axes.flat, NUMERIC):
    data = df[col].dropna()
    if col == "dist_to_road":
        data = np.log10(data + 1)
        col = f"log10({col})"
    ax.hist(data, bins=50, color=CLASS_COLORS["Alta"])
    ax.set_title(col)
for ax in axes.flat[len(NUMERIC):]:
    ax.remove()
plt.tight_layout(); plt.show()

- `elevation` y `wind_speed` se amontonan cerca de cero y tienen una cola larga a la derecha: la mayoría de plantas está en sitios bajos y con poco viento.
- `dist_to_road` solo se puede leer en escala logarítmica. La planta típica está a unos 700 m de una vía.
- `longitude` tiene tres picos, que corresponden a América, Europa y Asia.
- `latitude` se concentra en el hemisferio norte; solo el 4.6 % de las plantas está al sur del ecuador.
- `optimal_tilt` se concentra entre 28° y 37° (mediana 33.6°), con una cola hacia 0° de las plantas cercanas al ecuador. Su forma es casi la de `latitude`.
- `humidity` se carga hacia valores altos (70 % la mediana), y `ghi` y `ambient_temperature` tienen varios picos, reflejo de los distintos climas.
- `wind_direction` tiene dos picos: vientos del este-sureste (~100°) y del suroeste (~230°). Es un ángulo: 0° y 360° son la misma dirección.